# Prompt  

Dame un diseño de una clase sobre análisis de datos descriptivo con lenguaje sql gestionado desde pandas. Donde se plantee el problema de administración de empresas que se resuelva con el análisis de datos y donde tales datos se encuentre en una base de datos de archivo .db creada artificialmente con numpy, sql y pandas

Combinar SQL y Pandas es el "superpoder" de cualquier analista de datos moderno: aprovechas la eficiencia de SQL para filtrar y agregar grandes volúmenes de datos, y la flexibilidad de Pandas para el análisis y la manipulación final.



Aquí tienes una propuesta completa de diseño de clase, estructurada de forma didáctica y lista para impartir.



# Diseño de Clase: Análisis Descriptivo con SQL y Pandas para la Toma de Decisiones



**Nivel:** Intermedio (se asumen conocimientos básicos de Python y consultas `SELECT`).



**Objetivo de la clase:** Al finalizar, los estudiantes sabrán cómo conectar Pandas a una base de datos SQLite, ejecutar consultas SQL analíticas para resolver un problema de negocio y presentar métricas descriptivas clave.




## 1. El Problema de Negocio (Caso de Estudio)

> **Contexto:** "Café del Bosque" es una cadena de cafeterías con 3 sucursales. La dirección general nota que, aunque las ventas totales se mantienen, el margen de ganancias ha bajado en el último trimestre.
> **El Reto:** El Director de Operaciones necesita un informe analítico descriptivo urgente que responda:
> 1. ¿Cuál es el ticket promedio por sucursal?
> 2. ¿Qué categorías de productos generan más ingresos y cuáles están estancadas?
> 3. ¿En qué días y horas se concentra el mayor volumen de transacciones para optimizar el personal?
> 




## 2. Configuración del Entorno (Creación de Datos Artificiales)

Para que los estudiantes puedan trabajar, primero ejecutaremos un script oculto o preparatorio que crea la base de datos SQLite (`cafeteria.db`) utilizando **NumPy** para simular datos realistas y **Pandas/SQL** para estructurarlos.



In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path  # Librería ideal para manejar rutas

# 1. Configurar aleatoriedad para reproducibilidad
np.random.seed(42)

# 2. Crear datos maestros (Productos y Sucursales)
productos = {
    'id_producto': [101, 102, 103, 104, 105],
    'nombre': ['Café Espresso', 'Capuccino', 'Muffin de Arándanos', \
        'Panini de Pavo', 'Té Verde'],
    'categoria': ['Bebidas Calientes', 'Bebidas Calientes', 'Repostería', \
        'Alimentos', 'Bebidas Frías'],
    'precio': [2.50, 3.80, 4.00, 6.50, 3.00]
}
df_productos = pd.DataFrame(productos)

sucursales = {
    'id_sucursal': [1, 2, 3],
    'nombre_sucursal': ['Centro Histórico', 'Zona Norte', 'Centro Comercial']
}
df_sucursales = pd.DataFrame(sucursales)

# 3. Simular Transacciones (Datos de Ventas)
n_ventas = 1000
fechas = [datetime(2026, 3, 1) + timedelta(days=int(np.random.randint(0, 90)), 
                                         hours=int(np.random.randint(7, 21)), 
                                         minutes=int(np.random.randint(0, 60))) \
                                             for _ in range(n_ventas)]

df_ventas = pd.DataFrame({
    'id_transaccion': range(1001, 1001 + n_ventas),
    'fecha_hora': fechas,
    'id_sucursal': np.random.choice(df_sucursales['id_sucursal'], size=n_ventas, p=[0.5, 0.3, 0.2]),
    'id_producto': np.random.choice(df_productos['id_producto'], size=n_ventas, p=[0.3, 0.3, 0.15, 0.15, 0.1]),
    'cantidad': np.random.choice([1, 2, 3], size=n_ventas, p=[0.7, 0.2, 0.1])
})

# 4. Configurar la ruta específica y guardar la base de datos (.db)
# Definimos la ruta de la carpeta usando Path para evitar líos con 
# los slashes de Windows
ruta_carpeta = Path(r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\2_clases\unidad4\2_datos")

# Crear las carpetas si no existen (evita que el script falle si la ruta 
# es nueva)
ruta_carpeta.mkdir(parents=True, exist_ok=True)

# Definir la ruta completa del archivo
ruta_base_datos = ruta_carpeta / "cafeteria.db"

# Establecer la conexión usando la nueva ruta
conn = sqlite3.connect(str(ruta_base_datos))

df_productos.to_sql('productos', conn, if_exists='replace', index=False)
df_sucursales.to_sql('sucursales', conn, if_exists='replace', index=False)
df_ventas.to_sql('ventas', conn, if_exists='replace', index=False)

conn.close()

print(f"¡Base de datos 'cafeteria.db' creada con éxito en:\n{ruta_base_datos}")

¡Base de datos 'cafeteria.db' creada con éxito en:
C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\2_clases\unidad4\2_datos\cafeteria.db



## 3. Estructura del Script de Análisis (La Clase)

Una vez creada la base de datos, comienza la explicación paso a paso de cómo extraer valor de ella con SQL y Pandas.



# Paso 1: Conexión a la Base de Datos

Explicar a los alumnos cómo Pandas se conecta nativamente mediante la librería `sqlite3` (o `sqlalchemy`).



In [3]:
import pandas as pd
import sqlite3
from pathlib import Path

# 1. Definir la ruta exacta donde se encuentra el archivo .db
ruta_base_datos = Path(r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\2_clases\unidad4\2_datos\cafeteria.db")

# 2. Crear la conexión apuntando a esa ruta específica
# (Se transforma a string con str() ya que sqlite3 lo requiere así en algunas versiones)
conexion = sqlite3.connect(str(ruta_base_datos))

# --- A partir de aquí ya puedes usar 'conexion' con Pandas ---
# Ejemplo de uso:
# query = "SELECT * FROM productos;"
# df_productos = pd.read_sql_query(query, conexion)

# No olvides cerrar la conexión al final de tu análisis
# conexion.close()


# Paso 2: Análisis Descriptivo 1 – Ticket Promedio por Sucursal

*Concepto SQL a enseñar:* `JOIN`, `GROUP BY`, y funciones de agregación (`SUM`, `COUNT`).
*Concepto Pandas:* Uso de `pd.read_sql_query()` para traer data limpia y estructurada directamente como un DataFrame.



In [4]:
query_ticket = """
SELECT 
    s.nombre_sucursal,
    COUNT(v.id_transaccion) as total_transacciones,
    SUM(v.cantidad * p.precio) as ingresos_totales,
    SUM(v.cantidad * p.precio) / COUNT(DISTINCT v.id_transaccion) as ticket_promedio
FROM ventas v
JOIN sucursales s ON v.id_sucursal = s.id_sucursal
JOIN productos p ON v.id_producto = p.id_producto
GROUP BY s.nombre_sucursal;
"""

df_ticket = pd.read_sql_query(query_ticket, conexion)
print("--- TICKET PROMEDIO POR SUCURSAL ---")
print(df_ticket)


--- TICKET PROMEDIO POR SUCURSAL ---
    nombre_sucursal  total_transacciones  ingresos_totales  ticket_promedio
0  Centro Comercial                  198            1017.1         5.136869
1  Centro Histórico                  495            2537.5         5.126263
2        Zona Norte                  307            1552.0         5.055375



## Paso 3: Análisis Descriptivo 2 – Rendimiento por Categoría de Producto

*Concepto SQL a enseñar:* Análisis de participación de mercado interno.
*Concepto Pandas:* Manipulación del resultado, formateo de datos (moneda y porcentajes).



In [5]:
query_productos = """
SELECT 
    p.categoria,
    p.nombre as producto,
    SUM(v.cantidad) as unidades_vendidas,
    SUM(v.cantidad * p.precio) as ingresos_producto
FROM ventas v
JOIN productos p ON v.id_producto = p.id_producto
GROUP BY p.nombre
ORDER BY ingresos_producto DESC;
"""

df_productos_analisis = pd.read_sql_query(query_productos, conexion)

# Usamos Pandas para calcular el porcentaje de participación rápidamente
total_ingresos = df_productos_analisis['ingresos_producto'].sum()
df_productos_analisis['porcentaje_participacion'] = (df_productos_analisis['ingresos_producto'] / total_ingresos) * 100

print("\n--- RENDIMIENTO DE PRODUCTOS ---")
print(df_productos_analisis.round(2))



--- RENDIMIENTO DE PRODUCTOS ---
           categoria             producto  unidades_vendidas  \
0  Bebidas Calientes            Capuccino                382   
1          Alimentos       Panini de Pavo                204   
2  Bebidas Calientes        Café Espresso                440   
3         Repostería  Muffin de Arándanos                215   
4      Bebidas Frías             Té Verde                123   

   ingresos_producto  porcentaje_participacion  
0             1451.6                     28.43  
1             1326.0                     25.97  
2             1100.0                     21.54  
3              860.0                     16.84  
4              369.0                      7.23  



# Paso 4: Análisis Descriptivo 3 – Patrones de Tiempo (Días y Horas)

*Concepto SQL a enseñar:* Funciones de manejo de strings y fechas en SQL (`strftime` en SQLite) para agrupar temporalmente.


In [6]:
query_tiempo = """
SELECT 
    strftime('%H', fecha_hora) as hora,
    COUNT(id_transaccion) as volumen_ventas
FROM ventas
GROUP BY hora
ORDER BY hora;
"""

df_tiempo = pd.read_sql_query(query_tiempo, conexion)
print("\n--- VOLUMEN DE VENTAS POR HORA ---")
print(df_tiempo)

# Cerrar la conexión siempre al terminar
conexion.close()



--- VOLUMEN DE VENTAS POR HORA ---
   hora  volumen_ventas
0    07              80
1    08              72
2    09              69
3    10              68
4    11              77
5    12              62
6    13              76
7    14              69
8    15              75
9    16              58
10   17              63
11   18              81
12   19              78
13   20              72



## 4. Conclusión y Toma de Decisiones



Para cerrar la clase con enfoque administrativo, debemos guiar a los alumnos a interpretar los DataFrames resultantes:



1. **Hallazgo de Sucursales:** Si la sucursal "Zona Norte" tiene menos transacciones pero un ticket promedio mucho más alto, la estrategia debe ser atraer más clientes ahí.


2. **Optimización de Menú:** Si la categoría "Repostería" tiene baja participación, se puede proponer una estrategia de *cross-selling* (combo de Café + Muffin por un precio especial).


3. **Mano de Obra:** El análisis de horas indicará con precisión los picos de la mañana (ej. 8:00 AM) o tarde, permitiendo al administrador programar los turnos del personal de manera eficiente para reducir costos operativos.